# Inline 3D 检查 Single-Arm Grav 渲染

这个 notebook 参照 `inspect_json_robot_render_inline_no_side_predefined.ipynb`，用于单臂 FOAR 场景下检查 Flexiv Grav 夹爪的 inline 3D 渲染效果。

用途：
- 读取单臂 JSON 标定；
- 从真实 depth 重建点云并变换到 renderer 一致的 O3D 坐标；
- 使用 `left_robot_grav.urdf` 重建整臂 + Grav gripper mesh；
- 在 notebook 中 inline 显示点云、robot mesh、base/tcp/finger frame。


In [ ]:
from __future__ import annotations

import json
import h5py
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import yaml
from easydict import EasyDict as edict
from PIL import Image
from scipy.spatial.transform import Rotation as R
import trimesh

WORKSPACE_ROOT = Path('/home/haoxiang/rise2_mask_aware')
AIREXO_ROOT = WORKSPACE_ROOT / 'airexo'
for p in [WORKSPACE_ROOT, AIREXO_ROOT]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

from airexo.helpers.constants import (
    O3D_RENDER_TRANSFORMATION,
    ROBOT_PREDEFINED_TRANSFORMATION,
)
from airexo.helpers import urdf_robot as robot_helper


In [ ]:
# ===== 用户参数 =====
CALIB_JSON = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/calib/result.json')
CALIB_NPY = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/calib/rise2_calib_single_foar_purplebox.npy')
SCENE_DIR = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/val/scene_0010/cam_104422070117')
CAMERA_TIMESTAMP = 1774522305230
H5_PATH = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/val/scene_0010/lowdim/lowdim.h5')
ARM_SUFFIX = '062770'
LEFT_URDF = str((WORKSPACE_ROOT / 'airexo/airexo/urdf_models/zihao_single_worldbase_y_neg90/left_robot_grav.urdf').resolve())
JOINT_CFG_PATH = WORKSPACE_ROOT / 'airexo/airexo/configs/joint/left/robot.yaml'

JOINT_SOURCE_MODE = 'h5_frame_index'   # 'manual_deg' | 'h5_frame_index' | 'h5_nearest_timestamp'
H5_FRAME_INDEX = 5000
EXACT_H5_TIMESTAMP = False

INIT_JOINT_DEG = [39.877, -64.683, 27.603, 67.212, -43.0428, -4.469, -19.715]
INIT_GRIPPER_WIDTH = 0

DEPTH_SCALE = 1000.0
MIN_DEPTH_M = 0.05
MAX_DEPTH_M = 1.50
POINT_STRIDE = 6
POINT_MAX = 120000
MESH_SAMPLE_LIMIT = 20000
FRAME_AXIS_LEN = 0.08


In [ ]:
def invert_T(T):
    T = np.asarray(T, dtype=np.float64)
    out = np.eye(4, dtype=np.float64)
    out[:3, :3] = T[:3, :3].T
    out[:3, 3] = -T[:3, :3].T @ T[:3, 3]
    return out

def pose7_wxyz_to_mat(pose7):
    pose7 = np.asarray(pose7, dtype=np.float64).reshape(7)
    t = pose7[:3]
    qw, qx, qy, qz = pose7[3:]
    mat = np.eye(4, dtype=np.float64)
    mat[:3, :3] = R.from_quat([qx, qy, qz, qw]).as_matrix()
    mat[:3, 3] = t
    return mat

def load_json_pose(path: Path):
    data = json.loads(path.read_text())
    return pose7_wxyz_to_mat(data['pose_in_link'])

def load_intrinsic(path: Path):
    data = np.load(str(path), allow_pickle=True).item()
    return np.asarray(data['intrinsics']['104422070117'], dtype=np.float64)

def find_h5_index(h5_timestamps: np.ndarray, camera_timestamp: int, exact: bool = False):
    if exact:
        hit = np.where(h5_timestamps == int(camera_timestamp))[0]
        if len(hit) == 0:
            raise ValueError(f'exact timestamp {camera_timestamp} not found in h5')
        return int(hit[0])

    insert_idx = int(np.searchsorted(h5_timestamps, int(camera_timestamp)))
    if insert_idx <= 0:
        return 0
    if insert_idx >= len(h5_timestamps):
        return len(h5_timestamps) - 1
    left_idx = insert_idx - 1
    right_idx = insert_idx
    left_diff = abs(int(h5_timestamps[left_idx]) - int(camera_timestamp))
    right_diff = abs(int(h5_timestamps[right_idx]) - int(camera_timestamp))
    return left_idx if left_diff <= right_diff else right_idx

def extract_joint_from_h5_by_index(h5_path: Path, frame_index: int, arm_suffix: str):
    with h5py.File(h5_path, 'r') as f:
        h5_timestamps = np.asarray(f['timestamp'][:], dtype=np.int64)
        idx = int(frame_index)
        if idx < 0 or idx >= len(h5_timestamps):
            raise IndexError(f'h5 frame index out of range: {idx}, len={len(h5_timestamps)}')
        matched_ts = int(h5_timestamps[idx])
        joint7 = np.asarray(f[f'joint_position_rad_{arm_suffix}'][idx], dtype=np.float64)
        gripper = float(np.asarray(f[f'ee_state_{arm_suffix}'][idx]).reshape(-1)[0])
        tcp = np.asarray(f[f'tcp_pose_{arm_suffix}'][idx], dtype=np.float64) if f'tcp_pose_{arm_suffix}' in f else None
    joint = np.concatenate([joint7, [gripper]], axis=0)
    return idx, matched_ts, joint, tcp

def extract_joint_from_h5_by_timestamp(h5_path: Path, camera_timestamp: int, arm_suffix: str, exact_ts: bool):
    with h5py.File(h5_path, 'r') as f:
        h5_timestamps = np.asarray(f['timestamp'][:], dtype=np.int64)
        idx = find_h5_index(h5_timestamps, camera_timestamp, exact=exact_ts)
        matched_ts = int(h5_timestamps[idx])
        joint7 = np.asarray(f[f'joint_position_rad_{arm_suffix}'][idx], dtype=np.float64)
        gripper = float(np.asarray(f[f'ee_state_{arm_suffix}'][idx]).reshape(-1)[0])
        tcp = np.asarray(f[f'tcp_pose_{arm_suffix}'][idx], dtype=np.float64) if f'tcp_pose_{arm_suffix}' in f else None
    joint = np.concatenate([joint7, [gripper]], axis=0)
    return idx, matched_ts, joint, tcp

def summarize_frame_delta(h5_path: Path, arm_suffix: str, anchor_index: int, compare_index: int):
    with h5py.File(h5_path, 'r') as f:
        joint_ds = np.asarray(f[f'joint_position_rad_{arm_suffix}'][:], dtype=np.float64)
        ee_ds = np.asarray(f[f'ee_state_{arm_suffix}'][:], dtype=np.float64).reshape(len(joint_ds), -1)
    a = int(anchor_index)
    b = int(compare_index)
    joint_delta = joint_ds[b] - joint_ds[a]
    ee_delta = ee_ds[b] - ee_ds[a]
    return joint_delta, ee_delta

def init_joint_deg_to_joint(joint_deg, gripper_width):
    joint_deg = np.asarray(joint_deg, dtype=np.float64).reshape(7)
    return np.concatenate([np.deg2rad(joint_deg), [float(gripper_width)]], axis=0)

def json_base_to_cam_to_renderer_cam_to_base(T_base_to_cam):
    return invert_T(T_base_to_cam) @ invert_T(np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64))

def load_mesh_vertices_faces(mesh_rel_path: str, urdf_file: str):
    mesh_path = Path(urdf_file).parent / mesh_rel_path
    mesh = trimesh.load_mesh(mesh_path, process=False)
    if hasattr(mesh, 'geometry'):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    faces = np.asarray(mesh.faces, dtype=np.int32)
    return vertices, faces

def apply_transform(vertices: np.ndarray, T: np.ndarray):
    homo = np.concatenate([vertices, np.ones((vertices.shape[0], 1), dtype=np.float64)], axis=1)
    out = (T @ homo.T).T
    return out[:, :3]

def downsample_faces(faces: np.ndarray, limit: int):
    if faces.shape[0] <= limit:
        return faces
    idx = np.linspace(0, faces.shape[0] - 1, limit).astype(np.int64)
    return faces[idx]

def add_frame(fig, T, name, axis_len=0.08):
    origin = T[:3, 3]
    axes = T[:3, :3]
    colors = ['red', 'green', 'blue']
    labels = ['x', 'y', 'z']
    for i in range(3):
        p1 = origin
        p2 = origin + axes[:, i] * axis_len
        fig.add_trace(go.Scatter3d(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
            mode='lines',
            line=dict(color=colors[i], width=6),
            name=f'{name}_{labels[i]}',
            showlegend=False,
        ))

def find_nearest_image_timestamp(scene_dir: Path, target_timestamp: int):
    color_dir = scene_dir / 'color'
    ts_candidates = sorted(int(p.stem) for p in color_dir.glob('*.png'))
    if len(ts_candidates) == 0:
        raise FileNotFoundError(f'no color png found under {color_dir}')
    ts_arr = np.asarray(ts_candidates, dtype=np.int64)
    idx = int(np.argmin(np.abs(ts_arr - int(target_timestamp))))
    return int(ts_arr[idx])

def load_scene_images(scene_dir: Path, camera_timestamp: int):
    resolved_timestamp = find_nearest_image_timestamp(scene_dir, int(camera_timestamp))
    color_path = scene_dir / 'color' / f'{resolved_timestamp}.png'
    depth_path = scene_dir / 'depth' / f'{resolved_timestamp}.png'
    color = np.array(Image.open(color_path).convert('RGB'))
    depth = np.array(Image.open(depth_path))
    return color_path, depth_path, color, depth, resolved_timestamp

def depth_to_point_cloud(depth_mm: np.ndarray, intrinsic: np.ndarray):
    fx = intrinsic[0, 0]
    fy = intrinsic[1, 1]
    cx = intrinsic[0, 2]
    cy = intrinsic[1, 2]
    depth = depth_mm.astype(np.float64) / DEPTH_SCALE
    depth = depth[::POINT_STRIDE, ::POINT_STRIDE]
    h, w = depth.shape
    ys, xs = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    xs = xs * POINT_STRIDE
    ys = ys * POINT_STRIDE
    valid = np.isfinite(depth) & (depth > MIN_DEPTH_M) & (depth < MAX_DEPTH_M)
    z = depth[valid]
    x = (xs[valid] - cx) * z / fx
    y = (ys[valid] - cy) * z / fy
    pts = np.stack([x, y, z], axis=1)
    pts_h = np.concatenate([pts, np.ones((pts.shape[0], 1), dtype=np.float64)], axis=1)
    pts = (np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64) @ pts_h.T).T[:, :3]
    if pts.shape[0] > POINT_MAX:
        idx = np.linspace(0, pts.shape[0] - 1, POINT_MAX).astype(np.int64)
        pts = pts[idx]
    return pts

def fk_tcp_candidates(joint, joint_cfgs, urdf_file):
    tf_map = robot_helper.forward_kinematic_single(
        joint=np.asarray(joint, dtype=np.float32),
        joint_cfgs=joint_cfgs,
        is_rad=True,
        urdf_file=urdf_file,
        with_visuals_map=False,
    )
    out = {}
    for key in ['flange', 'grav_base_link', 'left_finger_tip', 'right_finger_tip']:
        if key in tf_map:
            out[key] = tf_map[key].matrix()
    return out


In [ ]:
T_base_to_cam = load_json_pose(CALIB_JSON)
cam_to_base = json_base_to_cam_to_renderer_cam_to_base(T_base_to_cam)
intrinsic = load_intrinsic(CALIB_NPY)

with open(JOINT_CFG_PATH, 'r', encoding='utf-8') as f:
    joint_cfg = edict(yaml.safe_load(f))

if JOINT_SOURCE_MODE == 'manual_deg':
    left_joint = init_joint_deg_to_joint(INIT_JOINT_DEG, INIT_GRIPPER_WIDTH)
    matched_idx = None
    matched_ts = None
    scene_timestamp = int(CAMERA_TIMESTAMP)
    tcp_h5 = None
    joint_delta_from_frame0 = None
    ee_delta_from_frame0 = None
elif JOINT_SOURCE_MODE == 'h5_frame_index':
    matched_idx, matched_ts, left_joint, tcp_h5 = extract_joint_from_h5_by_index(H5_PATH, H5_FRAME_INDEX, ARM_SUFFIX)
    scene_timestamp = int(matched_ts)
    joint_delta_from_frame0, ee_delta_from_frame0 = summarize_frame_delta(H5_PATH, ARM_SUFFIX, 0, matched_idx)
elif JOINT_SOURCE_MODE == 'h5_nearest_timestamp':
    matched_idx, matched_ts, left_joint, tcp_h5 = extract_joint_from_h5_by_timestamp(H5_PATH, CAMERA_TIMESTAMP, ARM_SUFFIX, EXACT_H5_TIMESTAMP)
    scene_timestamp = int(matched_ts)
    joint_delta_from_frame0, ee_delta_from_frame0 = summarize_frame_delta(H5_PATH, ARM_SUFFIX, 0, matched_idx)
else:
    raise ValueError(f'invalid JOINT_SOURCE_MODE: {JOINT_SOURCE_MODE}')

color_path, depth_path, color_img, depth_img, resolved_scene_timestamp = load_scene_images(SCENE_DIR, scene_timestamp)
point_cloud = depth_to_point_cloud(depth_img, intrinsic)
cur_transforms_left, visuals_map_left = robot_helper.forward_kinematic_single(
    joint=left_joint.astype(np.float32),
    joint_cfgs=joint_cfg,
    is_rad=True,
    urdf_file=LEFT_URDF,
    with_visuals_map=True,
)
left_fk_candidates = fk_tcp_candidates(left_joint, joint_cfg, LEFT_URDF)

print('color_path =', color_path)
print('depth_path =', depth_path)
print('point_cloud shape =', point_cloud.shape)
print('left links =', len(cur_transforms_left))
print('left fk candidates =', list(left_fk_candidates.keys()))
print('JOINT_SOURCE_MODE =', JOINT_SOURCE_MODE)
print('left_joint_used =', left_joint)
print('scene_timestamp_requested =', scene_timestamp)
print('scene_timestamp_resolved =', resolved_scene_timestamp)
if JOINT_SOURCE_MODE != 'manual_deg':
    print('H5_PATH =', H5_PATH)
    print('ARM_SUFFIX =', ARM_SUFFIX)
    print('matched_h5_index =', matched_idx)
    print('matched_h5_timestamp =', matched_ts)
    print('scene_timestamp =', scene_timestamp)
    print('timestamp_diff_ms =', abs(int(matched_ts) - int(CAMERA_TIMESTAMP)))
    print('joint_delta_from_frame0 =', joint_delta_from_frame0)
    print('ee_delta_from_frame0 =', ee_delta_from_frame0)
    print('tcp_h5 =', tcp_h5)


In [ ]:
fig = go.Figure()

if point_cloud.shape[0] > 0:
    fig.add_trace(go.Scatter3d(
        x=point_cloud[:, 0],
        y=point_cloud[:, 1],
        z=point_cloud[:, 2],
        mode='markers',
        marker=dict(size=1.0, color=point_cloud[:, 2], colorscale='Viridis', opacity=0.35),
        name='depth_point_cloud',
    ))

for link, transform in cur_transforms_left.items():
    for v in visuals_map_left[link]:
        if v.geom_param is None:
            continue
        verts, faces = load_mesh_vertices_faces(v.geom_param, LEFT_URDF)
        faces = downsample_faces(faces, MESH_SAMPLE_LIMIT)
        tf = (
            np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
            @ cam_to_base
            @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
            @ transform.matrix()
            @ v.offset.matrix()
        )
        verts_tf = apply_transform(verts, tf)
        is_gripper = any(key in str(link).lower() for key in ['finger', 'knuckle', 'robotiq', 'grav', 'outer_bar', 'inner_bar', 'finger_mount'])
        color = 'rgba(255,120,50,0.72)' if is_gripper else 'rgba(50,120,255,0.55)'
        fig.add_trace(go.Mesh3d(
            x=verts_tf[:, 0],
            y=verts_tf[:, 1],
            z=verts_tf[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=color,
            opacity=0.7 if is_gripper else 0.55,
            name=f'left::{link}',
            showscale=False,
            hoverinfo='name',
        ))

add_frame(fig, np.eye(4), 'camera', axis_len=FRAME_AXIS_LEN)
left_base_world = (
    np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
    @ cam_to_base
    @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
)
add_frame(fig, left_base_world, 'left_base', axis_len=FRAME_AXIS_LEN)
for name, T in left_fk_candidates.items():
    left_world = (
        np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
        @ cam_to_base
        @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
        @ T
    )
    add_frame(fig, left_world, f'left_{name}', axis_len=FRAME_AXIS_LEN * 0.8)

title_suffix = f'mode={JOINT_SOURCE_MODE}'
if matched_idx is not None:
    title_suffix += f', h5_idx={matched_idx}, h5_ts={matched_ts}'
title_suffix += f', scene_ts={resolved_scene_timestamp}'
fig.update_layout(
    title='Inline 3D Single Arm Grav: Depth Point Cloud + Robot Mesh | ' + title_suffix,
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', aspectmode='data'),
    width=1300,
    height=950,
    showlegend=False,
)
fig
